# BraTS 2024 - nnU-Net (2D) baseline

Prepares the raw BraTS-GLI volumes for **nnU-Net v2**, trains the `2d` configuration and compares it with the U-Net / Attention U-Net from `02_training.ipynb` on the **same patient split**.

nnU-Net does not use the preprocessed slice tensors. It reads full NIfTI volumes and the `2d` configuration trains on axial slices by itself.

Steps:
1. Paths and environment variables
2. Build `Dataset501_BraTSGLI` (hard links, no extra disk space) from `data_splits.json`
3. Plan and preprocess
4. Write `splits_final.json` with the project split
5. Train (run in a terminal, takes a long time)
6. Predict on the test set
7. Evaluate: global Dice per class, same definition as `Trainer` (restricted to the same z-range as the 2D slices)

`pip install nnunetv2` is required.

## 1. Paths and environment

In [ ]:
import os, json, shutil, subprocess
from pathlib import Path

DATA_DIR = Path(r"E:\abronikowski\data")
RAW_CASES = DATA_DIR / "raw" / "training_data1_v2"
SPLITS_JSON = DATA_DIR / "data_splits.json"

NNUNET_BASE = Path(r"E:\abronikowski\nnunet")
os.environ["nnUNet_raw"] = str(NNUNET_BASE / "raw")
os.environ["nnUNet_preprocessed"] = str(NNUNET_BASE / "preprocessed")
os.environ["nnUNet_results"] = str(NNUNET_BASE / "results")
for k in ("nnUNet_raw", "nnUNet_preprocessed", "nnUNet_results"):
    Path(os.environ[k]).mkdir(parents=True, exist_ok=True)

DATASET_ID = 501
DATASET_NAME = f"Dataset{DATASET_ID}_BraTSGLI"
DATASET_DIR = Path(os.environ["nnUNet_raw"]) / DATASET_NAME
PREPROCESSED_DIR = Path(os.environ["nnUNet_preprocessed"]) / DATASET_NAME

# Channel order matches 02_training: T1, T1CE, T2, FLAIR
MODALITIES = ["t1n", "t1c", "t2w", "t2f"]

splits = json.loads(SPLITS_JSON.read_text())
print({k: len(v) for k, v in splits.items()})
print("Raw cases:", sum(1 for _ in RAW_CASES.iterdir()))

## 2. Build the nnU-Net dataset

`train` + `validate` go to `imagesTr` / `labelsTr` (the validation part is selected by `splits_final.json` below).
`test` goes to `imagesTs` / `labelsTs` and is never seen during training.

Files are **hard-linked** (same drive, no admin rights, no extra space); if linking fails, they are copied.

In [ ]:
def link_or_copy(src: Path, dst: Path) -> None:
    if dst.exists():
        return
    try:
        os.link(src, dst)
    except OSError:
        shutil.copy2(src, dst)

def add_case(case: str, images_dir: Path, labels_dir: Path) -> None:
    src_dir = RAW_CASES / case
    for ch, mod in enumerate(MODALITIES):
        link_or_copy(src_dir / f"{case}-{mod}.nii.gz", images_dir / f"{case}_{ch:04d}.nii.gz")
    link_or_copy(src_dir / f"{case}-seg.nii.gz", labels_dir / f"{case}.nii.gz")

dirs = {n: DATASET_DIR / n for n in ("imagesTr", "labelsTr", "imagesTs", "labelsTs")}
for d in dirs.values():
    d.mkdir(parents=True, exist_ok=True)

train_cases = splits["train"] + splits["validate"]
for case in train_cases:
    add_case(case, dirs["imagesTr"], dirs["labelsTr"])
for case in splits["test"]:
    add_case(case, dirs["imagesTs"], dirs["labelsTs"])

dataset_json = {
    "channel_names": {"0": "T1", "1": "T1CE", "2": "T2", "3": "FLAIR"},
    "labels": {"background": 0, "NCR": 1, "ED": 2, "ET": 3, "RC": 4},
    "numTraining": len(train_cases),
    "file_ending": ".nii.gz",
}
(DATASET_DIR / "dataset.json").write_text(json.dumps(dataset_json, indent=2))
print(f"{DATASET_NAME}: {len(train_cases)} training cases, {len(splits['test'])} test cases")

## 3. Plan and preprocess (2D only)

Takes a while (resampling and normalization of all training volumes).

In [ ]:
import importlib.util, sys

if importlib.util.find_spec("nnunetv2") is None:
    raise ImportError("nnunetv2 is not installed in this kernel's environment: run `pip install nnunetv2` and restart the kernel")

# Entry points called through the kernel's own interpreter, so it works even if the Scripts folder is not on PATH
ENTRY_POINTS = {
    "nnUNetv2_plan_and_preprocess": "nnunetv2.experiment_planning.plan_and_preprocess_entrypoints:plan_and_preprocess_entry",
    "nnUNetv2_train": "nnunetv2.run.run_training:run_training_entry",
    "nnUNetv2_predict": "nnunetv2.inference.predict_from_raw_data:predict_entry_point",
}

def run(cmd: list[str]) -> None:
    print(">", " ".join(cmd))
    module, func = ENTRY_POINTS[cmd[0]].split(":")
    code = f"import sys; from {module} import {func}; sys.argv[0] = {cmd[0]!r}; {func}()"
    subprocess.run([sys.executable, "-c", code, *cmd[1:]], check=True, env=os.environ)

run(["nnUNetv2_plan_and_preprocess", "-d", str(DATASET_ID), "-c", "2d", "--verify_dataset_integrity"])

## 4. Use the project split

Replaces nnU-Net's default 5-fold split with a single fold: `train` for training, `validate` for validation. Train it as **fold 0**.

In [ ]:
splits_final = [{"train": splits["train"], "val": splits["validate"]}]
(PREPROCESSED_DIR / "splits_final.json").write_text(json.dumps(splits_final, indent=2))
print("Wrote", PREPROCESSED_DIR / "splits_final.json")

## 5. Training

Run this in a terminal, not in the notebook (it takes many hours). The default trainer runs 1000 epochs of 250 iterations; `nnUNetTrainer_50epochs` is a quick first test. The command is printed below.

In [ ]:
TRAINER = "nnUNetTrainer_50epochs"   # use "nnUNetTrainer" for the full 1000-epoch run
for k in ("nnUNet_raw", "nnUNet_preprocessed", "nnUNet_results"):
    print(f'$env:{k} = "{os.environ[k]}"')
print(f"nnUNetv2_train {DATASET_ID} 2d 0 -tr {TRAINER}")

## 6. Predict on the test set

In [ ]:
PRED_DIR = NNUNET_BASE / "predictions" / "test_2d"
PRED_DIR.mkdir(parents=True, exist_ok=True)

run(["nnUNetv2_predict", "-i", str(dirs["imagesTs"]), "-o", str(PRED_DIR),
     "-d", str(DATASET_ID), "-c", "2d", "-f", "0", "-tr", TRAINER,
     "-chk", "checkpoint_best.pth"])

## 7. Evaluation

Global Dice per class accumulated over all test voxels (same as `dice_from_stats` in `brain_mri.models.metrics`), restricted to the z-range from `data/patients_z_boundaries.json` so the number is comparable to the test Dice from `02_training.ipynb`.

In [ ]:
import numpy as np
import nibabel as nib

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
z_bounds = json.loads((PROJECT_ROOT / "data" / "patients_z_boundaries.json").read_text())

CLASS_NAMES = {1: "NCR", 2: "ED", 3: "ET", 4: "RC"}
inter = {c: 0.0 for c in CLASS_NAMES}
total = {c: 0.0 for c in CLASS_NAMES}
missing_bounds = 0

for case in splits["test"]:
    gt = np.asanyarray(nib.load(dirs["labelsTs"] / f"{case}.nii.gz").dataobj).astype(np.uint8)
    pr = np.asanyarray(nib.load(PRED_DIR / f"{case}.nii.gz").dataobj).astype(np.uint8)
    if case in z_bounds:
        lo, hi = z_bounds[case]   # inclusive, as in get_slice_indices
        gt, pr = gt[..., lo:hi + 1], pr[..., lo:hi + 1]
    else:
        missing_bounds += 1
    for c in CLASS_NAMES:
        g, p = gt == c, pr == c
        inter[c] += np.logical_and(g, p).sum()
        total[c] += g.sum() + p.sum()

scores = {name: round(2 * inter[c] / total[c], 4) if total[c] else 1.0 for c, name in CLASS_NAMES.items()}
scores["mean"] = round(sum(scores.values()) / len(CLASS_NAMES), 4)
print(f"nnU-Net 2d test Dice: {scores}   (cases without z-bounds: {missing_bounds})")

Compare `scores` with `test_dice` from `02_training.ipynb`.